# 4 Searching architectures, and what a search may conclude

`1_representation_and_head/04_search` ran one Optuna search and read it, and
`03_what_does_the_algorithm_know` compared what the samplers know. This notebook runs a
search with one of them, TPE, on the composition family of `01_gates_highways_moe` -- gate,
highway, dense and sparse mixture of experts -- on `solar_10_minutes`: 20 solar sites pooled
into one training set, a daily cycle that is deterministically zero every night. A pool of
different sites is at least plausibly something a router could divide up. The questions are
whether a search finds out, and what it may conclude once it has.

Four things come on top of lesson 1's search: a baseline named before the search (4.1), a
learning-rate schedule as a lever of its own (4.2), a capacity-matched control inside the
search space and a confirmation of the winners over several seeds on windows the search never
saw (4.4), and a pruner that stops weak trials early (4.5).

In [ ]:
import functools
import sys
import time
from pathlib import Path

import datasets
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import torch
from loguru import logger
from notebooktester import param
from optuna.trial import TrialState
from solar_10_minutes import load_solar
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from vectormesh.components import Gate, Highway, MoE, NeuralNet, SparseMoE
from vectormesh.components.metrics import MAE
from vectormesh.training import Trainer, TrainerSettings
from windowing import window_index

logger.remove()
logger.add(sys.stderr, level="WARNING")
datasets.disable_progress_bars()
# a search shows one progress bar; Optuna's line per trial stays off
optuna.logging.set_verbosity(optuna.logging.WARNING)

# the window 2_kernels_state_and_gates/03_decomposition_and_nbeats uses on solar power
W, H = 288, 144
EPOCHS = param(15, test=1)
# 36 trials: six mechanisms, and two more axes for the routed ones
N_TRIALS = param(36, test=2)
SEEDS = param(5, test=2)
# where a Trainer may write a checkpoint; nothing here saves one, so it is never made
LOGDIR = Path("modellogs").resolve()

## 4.1 The baseline, first

Same discipline every notebook in this course insists on: name the number a search has
to beat before running one.

In [ ]:
@functools.cache
def load_solar_windows(
    w: int = W, h: int = H, n_series: int = 20, stride: int = 6
) -> dict:
    """The split_scale_window of 2_kernels_state_and_gates/03_decomposition_and_nbeats on
    load_solar's output: per site the first 60% of the series is train, the next 20% val,
    the last 20% test, scaled with the train part's mean and std. Kept per window length,
    so a search that draws `w` (E2) cuts every length once."""
    series = load_solar(w, h, n_series=n_series)

    parts = {"train": [[], []], "val": [[], []], "test": [[], []]}
    for s in series:
        n = len(s)
        cuts = {
            "train": (0, int(0.6 * n)),
            "val": (int(0.6 * n), int(0.8 * n)),
            "test": (int(0.8 * n), n),
        }
        mu = s[: cuts["train"][1]].mean()
        sigma = s[: cuts["train"][1]].std().clamp(min=1e-6)
        for split, (a, b) in cuts.items():
            v = (s[a:b] - mu) / sigma
            win = v[window_index(len(v), w + h, stride)]
            parts[split][0].append(win[:, :w])
            parts[split][1].append(win[:, w:])
    return {k: (torch.cat(v[0]), torch.cat(v[1])) for k, v in parts.items()}


power = load_solar_windows()
xtr, ytr = power["train"]
xva, yva = power["val"]
xte, yte = power["test"]
print(f"train {tuple(xtr.shape)}   val {tuple(xva.shape)}   test {tuple(xte.shape)}")


def mae(y: torch.Tensor, yhat: torch.Tensor) -> float:
    return float((y - yhat).abs().mean())


def make_expert(hidden: int, expert_hidden: int) -> nn.Module:
    # a bottleneck 2-layer expert -- Linear -> GELU -> Linear -- with its own width
    # independent of the block's hidden dim. A single Linear (01_gates_highways_moe's
    # expert shape) can only ever affine-remap the shared feature space; a real
    # specialist needs at least one nonlinearity to represent a genuinely different
    # regime, which is the whole premise a router is betting on.
    return NeuralNet(hidden_size=[hidden, expert_hidden], out_size=hidden)


def make_model(
    mechanism: str,
    hidden: int,
    n_experts: int = 4,
    expert_hidden: int | None = None,
    w: int = W,
    h: int = H,
) -> nn.Module:
    # the family 01_gates_highways_moe built: Linear -> GELU -> [mechanism] -> Linear.
    # "plain" has nothing in the middle; "deeper" is the capacity-matched control of
    # "gate", and both are one more choice of the search space (4.3).
    expert_hidden = expert_hidden or hidden
    if mechanism == "plain":
        mid = nn.Identity()
    elif mechanism == "deeper":
        # the capacity-matched control of `gate`: a Linear -> GELU block with exactly
        # the parameters of Gate's weight matrix and bias, and no gating
        mid = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU())
    elif mechanism == "gate":
        mid = Gate(hidden_size=hidden)
    elif mechanism == "highway":
        mid = Highway(transform=NeuralNet(hidden, hidden), hidden_size=hidden)
    elif mechanism == "moe":
        mid = MoE(
            experts=[make_expert(hidden, expert_hidden) for _ in range(n_experts)],
            hidden_size=hidden,
            out_size=hidden,
        )
    elif mechanism == "sparse_moe":
        mid = SparseMoE(
            experts=[make_expert(hidden, expert_hidden) for _ in range(n_experts)],
            hidden_size=hidden,
            out_size=hidden,
            k=1,
        )
    else:
        raise ValueError(mechanism)
    return nn.Sequential(nn.Linear(w, hidden), nn.GELU(), mid, nn.Linear(hidden, h))


def param_counts(model: nn.Module, mechanism: str, k: int = 1) -> tuple[int, int]:
    """(total, active) parameter counts for one forward pass. Every mechanism here
    except sparse_moe runs every parameter on every input, so total == active --
    that equality is itself the point dense mechanisms cannot escape. sparse_moe only
    ever exercises k of its N experts per example: active counts the router, the
    block's other layers, and k experts' worth of weights, not all N."""
    total = sum(p.numel() for p in model.parameters())
    if mechanism != "sparse_moe":
        return total, total
    mid = model[2]
    per_expert = sum(p.numel() for p in mid.experts[0].parameters())
    n_experts = len(mid.experts)
    non_expert = total - n_experts * per_expert
    active = non_expert + k * per_expert
    return total, active


validloader = DataLoader(TensorDataset(xva, yva), batch_size=len(xva))


def run(
    model: nn.Module, epochs: int = EPOCHS, lr: float = 1e-3, batch_size: int = 64
) -> tuple[float, float]:
    """Train for `epochs` epochs without early stopping; return the val MAE after the last
    epoch and the test MAE of the same weights."""
    # one Trainer per run: progress=False drops the tqdm bars, and nothing is written
    # to LOGDIR, so constructing one per seed costs nothing
    trainloader = DataLoader(
        TensorDataset(xtr, ytr), batch_size=batch_size, shuffle=True
    )
    settings = TrainerSettings(
        epochs=epochs,
        metrics=[MAE()],
        logdir=LOGDIR,
        train_steps=len(trainloader),
        valid_steps=len(validloader),
        optimizer_kwargs={"lr": lr},
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.MSELoss(),
        optimizer=torch.optim.Adam,
        traindataloader=trainloader,
        validdataloader=validloader,
        progress=False,
    )
    val_mae = trainer.loop().metric_dict["MAE"]
    model.eval()
    with torch.no_grad():
        return val_mae, mae(yte, model(xte))


scores, tests = [], []
start = time.time()
for seed in range(SEEDS):
    torch.manual_seed(seed)
    val_mae, test_mae = run(make_model("plain", hidden=64))
    scores.append(val_mae)
    tests.append(test_mae)

BASELINE = float(np.mean(scores))
BASELINE_STD = float(np.std(scores))
BASELINE_TEST = float(np.mean(tests))
BASELINE_TEST_STD = float(np.std(tests))
print(f"seeds: {[f'{s:.4f}' for s in scores]}")
print(
    f"\nBASELINE (plain, hidden=64)  val {BASELINE:.4f} +/- {BASELINE_STD:.4f}   "
    f"test {BASELINE_TEST:.4f} +/- {BASELINE_TEST_STD:.4f}   "
    f"({time.time() - start:.0f}s for {SEEDS} runs)"
)
print(f"anything inside +/-{2 * BASELINE_STD:.4f} of that is not a result.")

## 4.2 The learning rate: coarse first, then fine

The baseline above holds `lr` at 1e-3 for every epoch, and the search below treats `lr` as
one more number to choose. But a learning rate is the size of the optimizer's step, and the
right size changes during a run. A big step gets near a good solution quickly and then keeps
bouncing around it; a small step settles precisely but needs many epochs to arrive. A
**learning-rate scheduler** lowers `lr` while training runs: the chainsaw first, for the
shape, the chisel after, for the detail. **Cosine decay** is the smoothest version: `lr`
starts at its maximum and follows half a cosine wave down to about zero at the last epoch.

The same idea returns in the search of 4.3 as a rule about *effort*: spend the first moves
wide and cheap, and the last ones narrow and precise.

In the Trainer a scheduler is one extra argument, `scheduler=torch.optim.lr_scheduler.CosineAnnealingLR`,
with `scheduler_kwargs={"T_max": EPOCHS}` for the length of the half cosine; it steps once per epoch.

Four arms, the same model as the baseline (`plain`, hidden=64), the same 15 epochs, the
same seeds, no early stopping, scored on the validation windows:

| arm | `lr` during training | what it isolates |
|---|---|---|
| `constant 1e-3` | 1e-3 at every epoch | the baseline above |
| `cosine 1e-3` | 1e-3, falling to about 0 | the decay alone: same start as the baseline |
| `constant 3e-3` | 3e-3 at every epoch | a bigger step, no decay (3e-3 is the top of the `lr` range the search below uses) |
| `cosine 3e-3` | 3e-3, falling to about 0 | the bigger start with the decay |

Each cosine arm has a constant arm with the same start, so every pair differs in the decay
only. Before you run it: for which start do you expect the decay to help most, and does the
bigger start (the chainsaw) end lower than the small one once both decay?

In [ ]:
def lr_run(lr: float, cosine: bool, seed: int) -> list[float]:
    """Train `plain` (hidden=64) for EPOCHS epochs starting at `lr`, with or without
    cosine decay to 0 over those epochs. Returns the val MAE after every epoch."""
    torch.manual_seed(seed)
    model = make_model("plain", hidden=64)
    trainloader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)
    val_curve: list[float] = []

    def track(epoch, train_loss, test_loss, metric_dict):
        val_curve.append(metric_dict["MAE"])

    settings = TrainerSettings(
        epochs=EPOCHS,
        metrics=[MAE()],
        logdir=LOGDIR,
        train_steps=len(trainloader),
        valid_steps=len(validloader),
        optimizer_kwargs={"lr": lr},
        scheduler_kwargs={"T_max": EPOCHS},
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.MSELoss(),
        optimizer=torch.optim.Adam,
        scheduler=torch.optim.lr_scheduler.CosineAnnealingLR if cosine else None,
        traindataloader=trainloader,
        validdataloader=validloader,
        reporters=[track],
        progress=False,
    )
    trainer.loop()
    return val_curve


def lr_schedule(lr: float, cosine: bool) -> list[float]:
    """The lr of every epoch: constant, or CosineAnnealingLR's half cosine from `lr`
    to 0 (the scheduler steps once per epoch, after it)."""
    if not cosine:
        return [lr] * EPOCHS
    return [lr * 0.5 * (1 + np.cos(np.pi * e / EPOCHS)) for e in range(EPOCHS)]


LR_ARMS = {
    "constant 1e-3": (1e-3, False),
    "cosine 1e-3": (1e-3, True),
    "constant 3e-3": (3e-3, False),
    "cosine 3e-3": (3e-3, True),
}
lr_results: dict[str, dict] = {}
for name, (lr0, cosine) in LR_ARMS.items():
    curves = np.array([lr_run(lr0, cosine, seed) for seed in range(SEEDS)])
    lr_results[name] = {"curves": curves, "lr": lr_schedule(lr0, cosine)}
    last = curves[:, -1]
    print(
        f"{name:14s} val MAE at epoch {EPOCHS}: {last.mean():.4f} +/- {last.std():.4f}   "
        f"best epoch of the mean curve: {curves.mean(0).argmin() + 1}"
    )

In [ ]:
fig, (ax_lr, ax_mae) = plt.subplots(1, 2, figsize=(11, 3.6))
epochs_axis = np.arange(1, EPOCHS + 1)
for name, r in lr_results.items():
    ax_lr.plot(epochs_axis, r["lr"], marker="o", ms=3, label=name)
    mean, std = r["curves"].mean(0), r["curves"].std(0)
    ax_mae.plot(epochs_axis, mean, label=name)
    ax_mae.fill_between(epochs_axis, mean - std, mean + std, alpha=0.2)
ax_lr.set_yscale("log")
ax_lr.set_xlabel("epoch")
ax_lr.set_ylabel("lr")
ax_lr.set_title("the learning rate of each arm")
ax_mae.set_xlabel("epoch")
ax_mae.set_ylabel("val MAE (mean, band = 1 std over seeds)")
ax_mae.set_title("what the lr does to the curve")
ax_mae.legend(fontsize=8)
plt.tight_layout()

Read each pair against the seed spread (at most 0.002):

- **Decay alone, at the baseline's start (`constant 1e-3` against `cosine 1e-3`):** 0.2281 ± 0.0020
  against 0.2165 ± 0.0003. A gain of 0.0116 from changing nothing but the schedule, about six
  times the baseline's own spread.
- **Decay with the bigger start (`constant 3e-3` against `cosine 3e-3`):** 0.2421 ± 0.0019
  against 0.2204 ± 0.0007, a gain of 0.0217. The constant 3e-3 curve never settles: it wobbles
  around 0.24 from the first epoch to the last, while the decaying curve falls steadily as
  its `lr` falls. A large step finds the neighbourhood and then keeps stepping over the
  bottom of it; the small steps at the end are what resolve the detail.
- **The bigger start did not pay at this budget.** `cosine 1e-3` (0.2165) ends below
  `cosine 3e-3` (0.2204), a gap of 0.0039 against spreads of 0.0003 and 0.0007. Fifteen epochs
  are enough for the small start to arrive, so the chainsaw has no time to save. Whether a
  bigger start wins with a larger epoch budget is not measured here (E5).
- **Both decaying curves are still falling at epoch 15** (their best epoch is the last), while
  the constant curves stopped improving; `cosine` ends at a fixed epoch because `T_max` is set
  to it, so it cannot be combined with early stopping on a free-running budget the way
  the runs in `01_gates_highways_moe` are.

Two consequences for what follows. The baseline of 4.1 (0.2281) is the number to beat at a
constant `lr`, and a schedule alone moves `plain` by 0.0116, a gain of the same order as the
differences between mechanisms that 4.4 reports. The search below keeps `lr` constant in
every trial, so each trial is compared with the others at the same `lr` policy, and none of
them has the decay.

## 4.3 The search space

`mechanism` is the axis this notebook exists to test: `plain` (no block in the middle),
`deeper` (a `Linear -> GELU` block with exactly the parameters of `gate` and no gating: the
capacity-matched control, folded into the search as one more choice), `gate`, `highway`,
`moe` and `sparse_moe`. `hidden` and `lr` are in the space because width and learning rate
can confound a comparison between architectures if they are not searched alongside it.
`n_experts` and `expert_hidden` only matter for `moe` and `sparse_moe`. Optuna can draw an
axis only when another axis has a given value, but a `Space` of `hpo` draws every axis in
every trial, so the other mechanisms ignore these two (their trials differ in a number that
does nothing). An expert is a `Linear -> GELU -> Linear` network with its own width
`expert_hidden`, so the search can trade many small experts against few large ones.

**The sampler.** `TPESampler`, the TPE of `03_what_does_the_algorithm_know`: its first
`n_startup_trials=10` trials are drawn at random; after that it splits the trials so far into
a good pile and a bad pile and draws the next configuration where the good ones are dense
compared with the bad ones. It learns from earlier trials, so it spends more of the later
trials on the mechanisms that scored well early, a point that matters when the results are
read (4.4). `seed=0` makes it propose the same configurations on every run of this notebook,
as long as the study starts empty: a second run adds its trials to the same study, TPE then
learns from the first run's trials too, and its numbers differ from the ones in this text.

**What a trial reports.** After every epoch a trial reports its val MAE to Optuna with
`trial.report`, so the study holds the whole curve of every trial, not only its end. The
study maximises its value, so the value is minus the MAE: higher is better, and the tables
below undo the minus. The curves are what the pruner of 4.5 decides on.

**Why this might not help, and what would make it help.** Routing only pays for its
parameters if the data contains distinct situations that benefit from different transforms:
a cloudy-Tuesday site against a clear-sky one. Pooling 20 sites is at least *plausibly* that
kind of heterogeneity, unlike a single homogeneous series. Whether it pays is what the search
checks, not what this paragraph assumes.

**The trade-off the search reports.** Every non-routed mechanism trains and uses the same
number of parameters. `sparse_moe` can *train* far more than it *uses* on one input (4.4
reports both numbers), the bet `01_gates_highways_moe` measures in 1.4 and 1.5.

**The budget.** 36 trials of 15 epochs each is 540 epochs of training. A trial's epochs are
fixed rather than early-stopped: a search ranks configurations under an equal short budget,
which is a legitimate economy; the numbers reported afterwards are confirmed in 4.4. The study
is `lesson3-search` in the course's Optuna store.

In [ ]:
from hpo.measure import Curves
from hpo.record import SESSION, RunRecord, StudyLog
from hpo.search import Choice, Float, Space

SPACE = Space(
    {
        "mechanism": Choice(
            ["plain", "deeper", "gate", "highway", "moe", "sparse_moe"]
        ),
        "hidden": Choice([32, 64, 128]),
        "n_experts": Choice([2, 4, 6, 8, 10, 16]),  # ignored unless moe/sparse_moe
        "expert_hidden": Choice([16, 32, 64]),  # ignored unless moe/sparse_moe
        "lr": Float(1e-4, 3e-3, log=True),
        "w": W,
        "h": H,
        "epochs": EPOCHS,
    }
)


def objective(trial: optuna.Trial, log: StudyLog) -> float:
    """One trial: draw a config from SPACE, train it for its epochs, report the val MAE
    after every epoch, and return it after the last. The study maximises its value, so
    the value is minus the MAE: higher is better. After every report the pruner may stop
    the trial (4.5); without one, every trial runs all its epochs."""
    config = SPACE.draw(trial)
    data = load_solar_windows(config["w"], config["h"])
    xtr, ytr = data["train"]
    xva, yva = data["val"]

    torch.manual_seed(0)
    model = make_model(
        config["mechanism"],
        config["hidden"],
        config["n_experts"],
        config["expert_hidden"],
        config["w"],
        config["h"],
    )
    total_params, active_params = param_counts(model, config["mechanism"])
    curves = Curves()
    val_mae: list[float] = []

    # one report per epoch, step = the epochs trained so far: a pruner needs the partial
    # curve of every trial to decide which ones to stop
    def to_optuna(epoch, train_loss, test_loss, metric_dict):
        curves.add(train_loss, test_loss, None)
        val_mae.append(metric_dict["MAE"])
        trial.report(-metric_dict["MAE"], epoch + 1)
        if trial.should_prune():
            raise optuna.TrialPruned()

    trainloader = DataLoader(TensorDataset(xtr, ytr), batch_size=64, shuffle=True)
    validloader = DataLoader(TensorDataset(xva, yva), batch_size=len(xva))
    settings = TrainerSettings(
        epochs=config["epochs"],
        metrics=[MAE()],
        logdir=LOGDIR,
        train_steps=len(trainloader),
        valid_steps=len(validloader),
        optimizer_kwargs={"lr": config["lr"]},
        earlystop_kwargs=None,
    )
    trainer = Trainer(
        model=model,
        settings=settings,
        loss_fn=nn.MSELoss(),
        optimizer=torch.optim.Adam,
        traindataloader=trainloader,
        validdataloader=validloader,
        reporters=[to_optuna],
        progress=False,  # the study's own bar shows progress across trials
    )
    start = time.time()
    try:
        trainer.loop()
    finally:
        # a stopped trial is recorded too: what it drew, and how far it got
        log.record(
            trial,
            RunRecord(
                role="trial",
                dataset="solar_10_minutes",
                config=config,
                fixed=SPACE.fixed(),
                train_seconds=time.time() - start,
                curves=curves,
                extra={
                    "val_mae": round(val_mae[-1], 5),
                    "epochs_trained": len(val_mae),
                    "total_params": total_params,
                    "active_params": active_params,
                },
            ),
        )
    return -val_mae[-1]


def search(
    name: str, pruner: optuna.pruners.BasePruner | None = None
) -> tuple[StudyLog, float]:
    """The same search every time, so two searches differ only in `pruner`: TPE with
    10 random starting trials and seed 0, N_TRIALS trials in all. Returns the study and
    the seconds the trials took."""
    log = StudyLog.search(
        name=name,
        space=SPACE,
        sampler=optuna.samplers.TPESampler(n_startup_trials=10, seed=0),
        pruner=pruner,
        attrs={"origin": "3_scaling_and_search/04_search", "value": "minus val MAE"},
    )
    log.study.set_metric_names(["minus val MAE"])
    start = time.time()
    log.optimize(lambda trial: objective(trial, log), n_trials=N_TRIALS)
    return log, time.time() - start


def trials_frame(log: StudyLog) -> pd.DataFrame:
    """One row per trial of this run of the notebook: what it drew, its val MAE after
    the last epoch it trained, and how many epochs that was."""
    rows = []
    for t in log.trials(TrialState.COMPLETE, TrialState.PRUNED):
        if t.user_attrs.get("session") != SESSION:
            continue
        config = t.user_attrs["config"]
        rows.append(
            {
                "mae": t.user_attrs["val_mae"],
                **{
                    k: config[k]
                    for k in ("mechanism", "hidden", "n_experts", "expert_hidden", "lr")
                },
                "total_params": t.user_attrs["total_params"],
                "active_params": t.user_attrs["active_params"],
                "epochs": t.user_attrs["epochs_trained"],
                "pruned": t.state == TrialState.PRUNED,
                "trial": t.number,
            }
        )
    return pd.DataFrame(rows)


full, full_seconds = search("lesson3-search")
df = trials_frame(full)
print(f"{len(df)} trials of {EPOCHS} epochs in {full_seconds:.0f}s")

## 4.4 Reading the result

In [ ]:
print(
    df.drop(columns=["epochs", "pruned", "trial"])
    .sort_values("mae")
    .head(8)
    .to_string(index=False)
)

print(f"\n{'mechanism':<12} {'trials':>7} {'best':>8} {'median':>8} {'worst':>8}")
print("-" * 47)
for name, group in df.groupby("mechanism"):
    print(
        f"{name:<12} {len(group):>7} {group['mae'].min():>8.4f} "
        f"{group['mae'].median():>8.4f} {group['mae'].max():>8.4f}"
    )
print(
    f"\nbaseline (plain, hidden=64, {SEEDS} seeds), val: "
    f"{BASELINE:.4f} +/- {BASELINE_STD:.4f}"
)

# what sparse_moe trains against what it uses on one input
sparse_trials = df[df["mechanism"] == "sparse_moe"].sort_values("mae")
if len(sparse_trials):
    best_sparse = sparse_trials.iloc[0]
    print(
        f"\nbest sparse_moe trial: n_experts={int(best_sparse['n_experts'])}, "
        f"expert_hidden={int(best_sparse['expert_hidden'])}, mae={best_sparse['mae']:.4f}\n"
        f"  params trained: {int(best_sparse['total_params']):>8,}\n"
        f"  params used per input: {int(best_sparse['active_params']):>8,}  "
        f"({best_sparse['active_params'] / best_sparse['total_params']:.0%} of the trained)"
    )

The search table, trial by trial: `moe` (hidden 128, 4 experts of width 16) has the lowest
val MAE, 0.2076, then `highway` at 0.2080. The best trial of each mechanism lies between
0.2076 and 0.2187, and every one of them beats the baseline of 4.1 (0.2281 ± 0.0020).

Three things stop this table from being read as a ranking of mechanisms.

- **The mechanisms got unequal numbers of trials**: 12 for `moe`, 9 for `highway`, 4 each
  for `plain`, `deeper` and `gate`, 3 for `sparse_moe`. TPE puts more of the later trials
  where the earlier ones scored well, so the mechanism with the most draws has the most
  chances at a low minimum, and the medians (`moe` 0.2095, `highway` 0.2111, the rest about
  0.219 to 0.221) are tilted in the same way.
- **Each trial is one seed.** The gaps between the best trials are a few thousandths, the
  size of the seed spread in 4.1.
- **The baseline is not the right control.** It is `plain` at hidden 64 and a constant `lr`
  of 1e-3, while the search also tunes width and `lr`: `plain`'s own best trial (0.2187)
  already beats the baseline by 0.009. The gain over the baseline is mostly tuning, not
  mechanism; the distance to `plain` at its own best, and to `deeper`, is what a mechanism
  may claim.

`sparse_moe`'s trade-off, made concrete: its best trial has 2 experts of width 64 and trains
88,978 parameters but uses 72,402 (81%) on any one input. That is a modest decoupling, since
with `k=1` and only 2 experts half of the routed capacity is active by construction.

### The score is not the only axis

"The best trial" is half a question: best at what cost? The same trials, scored on both axes
-- val MAE and the parameters that were trained -- form a cloud, and a trial is only worth
keeping if no other trial is at least as good on both. Those trials are the **Pareto
front**.

In [ ]:
def pareto_front(points: np.ndarray) -> np.ndarray:
    """Indices of the rows that no other row beats or ties on both columns (lower is
    better in both)."""
    keep = []
    for i, p in enumerate(points):
        dominated = np.any(np.all(points <= p, axis=1) & np.any(points < p, axis=1))
        if not dominated:
            keep.append(i)
    return np.array(keep)


points = df[["mae", "total_params"]].to_numpy(dtype=float)
front = pareto_front(points)
front_rows = df.iloc[front].sort_values("total_params")

fig, ax = plt.subplots(figsize=(8, 3.8))
for name, group in df.groupby("mechanism"):
    ax.scatter(group["total_params"], group["mae"], label=name, s=28, alpha=0.8)
ax.step(front_rows["total_params"], front_rows["mae"], where="post", color="k", lw=1.2)
ax.set_xscale("log")
ax.set_xlabel("parameters trained (log)")
ax.set_ylabel("val MAE")
ax.set_title("every trial; the black line is the Pareto front")
ax.legend(fontsize=7)
plt.tight_layout()
print(front_rows[["mae", "mechanism", "hidden", "total_params"]].to_string(index=False))

Six trials form the front, from `gate` at hidden 32 (15,056 parameters, val MAE 0.2197)
through `highway` (17,232; 0.2111) and `moe` (20,630; 0.2109) up to `moe` at hidden 128
(73,044; 0.2076). The last step is the instructive one: the trial with 31,680 parameters
(0.2082) is within 0.0006 of the lowest MAE of the whole search, at 43% of its parameters.
The front is nearly flat beyond 20,000 to 30,000 parameters, so "the best trial" is a
different answer depending on what a parameter costs you: the 73,044-parameter `moe` if
accuracy is all that counts, the 31,680-parameter one if a model of that size is cheaper to
ship, and a question that is incomplete until the second axis is named.

Two cautions. Every point is a single seed, so a trial just behind the front is not
reliably worse than one on it. And parameters are only one cost: `01_gates_highways_moe`
showed that a sparse layer with few parameters in use can still be slower per forward pass
than a dense one, so latency would be a third axis that this plot does not have.

### Confirming the winners

The search table compares the *best trial* of each mechanism, and that comparison has three
weaknesses: a mechanism that got more trials has more chances to draw a lucky one, every
trial is a single seed, and the winners were chosen on val, so their val scores are the
maxima of noisy numbers. The confirmation below takes the best trial of each mechanism,
trains its configuration again over `SEEDS` seeds, and scores it on the **test windows**,
which neither the search nor any choice has seen. The baseline row is the one from 4.1.

In [ ]:
best_per_mechanism = df.loc[df.groupby("mechanism")["mae"].idxmin()].set_index(
    "mechanism"
)

confirmed: dict[str, dict] = {}
for name, row in best_per_mechanism.iterrows():
    vals, tests = [], []
    for seed in range(SEEDS):
        torch.manual_seed(seed)
        model = make_model(
            name, int(row["hidden"]), int(row["n_experts"]), int(row["expert_hidden"])
        )
        v, t = run(model, lr=float(row["lr"]))
        vals.append(v)
        tests.append(t)
    confirmed[name] = {
        "val": (float(np.mean(vals)), float(np.std(vals))),
        "test": (float(np.mean(tests)), float(np.std(tests))),
        "params": int(row["total_params"]),
        "hidden": int(row["hidden"]),
    }

print(f"{'mechanism':<12} {'hidden':>6} {'params':>9} {'val MAE':>17} {'test MAE':>17}")
print("-" * 66)
print(
    f"{'baseline':<12} {64:>6} {'':>9} {BASELINE:>9.4f} +/-{BASELINE_STD:.4f} "
    f"{BASELINE_TEST:>9.4f} +/-{BASELINE_TEST_STD:.4f}"
)
for name, r in sorted(confirmed.items(), key=lambda kv: kv[1]["test"][0]):
    print(
        f"{name:<12} {r['hidden']:>6} {r['params']:>9,} "
        f"{r['val'][0]:>9.4f} +/-{r['val'][1]:.4f} "
        f"{r['test'][0]:>9.4f} +/-{r['test'][1]:.4f}"
    )

Read within a column; the test windows are a later stretch of each series and score higher
for every model, the baseline included (0.2607 against 0.2281).

- **`gate` beats its capacity-matched control.** `gate` and `deeper` both have 72,080
  parameters and both got 4 trials in the search, and on test `gate` scores 0.2382 ± 0.0008
  against 0.2437 ± 0.0017: a gap of 0.0055, about three times the larger spread, and the
  same direction on val (0.2085 against 0.2105). This is the first place in this lesson
  where a mechanism clearly beats a plain layer of the same size.
- **The lead of `moe` and `highway` is gone.** On test `highway` (0.2429), `deeper` (0.2437)
  and `moe` (0.2441 ± 0.0045) tie: the table of 4.4's search showed `moe` and `highway` in
  front because they had the most draws, not because they are better.
- **`sparse_moe` is no better than nothing.** 0.2507 ± 0.0025 against `plain`'s
  0.2510 ± 0.0014, with 88,978 parameters against 55,568.
- **The gain over the baseline is mostly tuning.** `plain` at hidden 128 and its own `lr`
  already reaches 0.2510 against the baseline's 0.2607; `gate` adds another 0.013 on top.

So the search proposed, and the seeds and the unseen windows decided: of five candidate
mechanisms, one survived the control, and not the one the search table liked best. One
more data point is not a law: on the electricity window of `01_gates_highways_moe` the gate
tied its control, and this is one series with one set of 20 sites.

## 4.5 Stopping the hopeless trials early

Every trial above ran its 15 epochs, including the ones that were clearly behind after
three. A **pruner** reads the partial curves the trials report and stops a trial that is
behind. `SuccessiveHalvingPruner` does it in rungs: after `min_resource=3` epochs a trial
goes on only if its score is in the best third (`reduction_factor=3`) of all trials that have
reached epoch 3 so far, and after 9 epochs the same again among those that reached 9. A trial
that survives both runs its 15 epochs. The trials run one after another, so "so far" matters:
the first trials compete against few others and are rarely stopped, the later ones against
many. The first effort is spent wide and cheap, on every configuration, and the last one
narrow and precise, on the survivors -- the shape of a learning-rate schedule from 4.2,
applied to the search instead of to one run.

Two checks. First whether the premise holds on the trials of 4.3: does the ranking after 3
epochs predict the ranking after 15? If it does, stopping early loses little; if a slow
starter often wins, the pruner discards exactly the trials that matter. Second, the same
search with the pruner switched on, as the study `lesson3-search-pruned`: the same space and
the same TPE with the same seed. The two searches start with the same random trials, but TPE
learns from every trial, a stopped one by the score it had when it stopped, so once the first
trial is stopped the two searches propose different configurations.

In [ ]:
GRACE = min(3, EPOCHS)  # the first epoch after which the pruner may stop a trial

# a trial reports minus the MAE at step = epochs trained; the minus is undone here
at_grace = pd.Series(
    {t: -full.study.trials[t].intermediate_values[GRACE] for t in df["trial"]}
)
at_end = pd.Series(
    {t: -full.study.trials[t].intermediate_values[EPOCHS] for t in df["trial"]}
)
rank_corr = at_grace.corr(at_end, method="spearman")
winner = int(at_end.idxmin())
winner_rank = int(at_grace.rank().loc[winner])
print(
    f"rank correlation of val MAE after {GRACE} epochs and after {EPOCHS}: {rank_corr:.2f}"
)
print(
    f"the final winner was number {winner_rank} of {len(at_grace)} after {GRACE} epochs "
    f"(the best third: {len(at_grace) // 3})"
)

In [ ]:
pruner = optuna.pruners.SuccessiveHalvingPruner(min_resource=GRACE, reduction_factor=3)
pruned, pruned_seconds = search("lesson3-search-pruned", pruner=pruner)
df_pruned = trials_frame(pruned)

finished = df_pruned[~df_pruned["pruned"]]
print(f"{'':16s} {'no pruner':>10} {'pruner':>10}")
print(f"{'seconds':16s} {full_seconds:>10.0f} {pruned_seconds:>10.0f}")
print(
    f"{'epochs trained':16s} {int(df['epochs'].sum()):>10d} {int(df_pruned['epochs'].sum()):>10d}"
)
print(
    f"{'trials stopped':16s} {int(df['pruned'].sum()):>10d} {int(df_pruned['pruned'].sum()):>10d}"
    f"   (of {len(df_pruned)})"
)
best_full, best_pruned = df["mae"].min(), finished["mae"].min()
print(f"{'best val MAE':16s} {best_full:>10.4f} {best_pruned:>10.4f}")
print(
    "epochs a trial of the pruned search ran:",
    df_pruned["epochs"].value_counts().sort_index().to_dict(),
)
top = finished.loc[finished["mae"].idxmin()]
print(
    f"best trial of the pruned search: {top['mechanism']}, hidden={int(top['hidden'])}, "
    f"lr={top['lr']:.1e}, {int(top['epochs'])} epochs"
)
print(
    "mechanisms of the trials that ran all epochs:",
    finished["mechanism"].value_counts().to_dict(),
)

**Does the premise hold?** The rank correlation between the val MAE after 3 epochs and
after 15 is 0.44: a weak guide. The trial that finished first was only 18th of 36 after 3
epochs, outside the best third (12) that ASHA keeps. On these trials a good early rank is a
loose signal, and a slow starter can win.

**What ASHA bought.** It stopped 28 of 36 trials and trained 240 epochs instead of 540 (44%),
and the best trial it found, a `highway` with hidden 128, reaches 0.2107 against the full
search's 0.2076. A gap of 0.003 is the size of a single seed's noise at this scale, so the
run does not show that the early stops cost anything, nor that they were free. Wall time
fell less than epochs (173 s against 213 s) because every trial loads its data before its
first epoch, a fixed cost that stopping early does not remove.

**The budget question.** The 300 epochs that ASHA saved would pay for 20 more full trials,
or for about 45 more at ASHA's average cost of under 7 epochs per trial. For a fixed budget the
choice is therefore between 36 full trials and more than twice as many, most of them stopped
after three epochs. Which wins depends on how well three epochs predict fifteen, which here
they do only weakly (E6).

## 4.6 Where the searches are

Both searches are studies in the course's Optuna store, the one every search of this course
writes to: `lesson3-search` with every trial run to the end, `lesson3-search-pruned` with
the pruner. Every trial carries what it drew (`config`), its val MAE after the last epoch it
ran (`val_mae`), how many epochs that was (`epochs_trained`), its parameters trained and used,
and its curve per epoch. `make optuna-dashboard` opens the store: the history plot of a study
shows the trials in order, and the intermediate-values plot shows the curve of every trial,
the stopped ones ending at epoch 3 or 9.

In [ ]:
for log in (full, pruned):
    states = pd.Series([t.state.name for t in log.trials()]).value_counts().to_dict()
    print(f"{log.name:22s} {len(log.trials()):>3} trials in the store  {states}")
print(f"\nthe store: {full.store.db}")

## What to take from this

- **Name the baseline first, and the right one.** 4.1 fixed the number to beat before any
  trial ran. The search then beat it with every mechanism, `plain` included, because it also
  tunes width and `lr`: a search result is read against a control that had the same tuning.
- **A schedule is a lever of its own.** `plain` at a constant `lr` of 1e-3 scores 0.2281; the
  same model with cosine decay from the same start scores 0.2165 (4.2). Coarse first, then
  fine is also the shape of the search: wide and cheap first, narrow and precise after.
- **The search proposes; seeds and unseen data decide.** The best trial of a search is the
  maximum of noisy numbers, drawn unequally often from each mechanism. Retrained over seeds on
  the test windows, the lead of `moe` and `highway` disappeared and `gate` stood out against its
  control (4.4).
- **The best trial depends on the second axis.** The Pareto front over val MAE and parameters
  shows how little the last 40,000 parameters bought (4.4).
- **A scheduler spends effort wide, then narrow.** ASHA stopped most trials after a few
  epochs and trained 44% of the epochs, with a rank correlation of 0.44 between early and
  late scores as the price of the premise (4.5).
- **Parameters trained and parameters used are different numbers for `sparse_moe`, and the
  same number for everything else.** The best `sparse_moe` trial trains 88,978 and uses 72,402
  per input (4.4).

## Exercises

**E1 -- predict before you read 4.4.** Given `01_gates_highways_moe`'s result on an
electricity window (nothing beat a plain layer of the same size), decide what you expect here,
on a series with a deterministically zero night and 20 pooled sites to route on. Which
mechanism do you expect to survive the confirmation, and does `sparse_moe` change your
prediction now that it can search narrower experts?

**E2 -- widen the space.** Add `w` to the search space (`tune.choice([144, 288, 576])`)
instead of holding it fixed. Does the best window interact with the best mechanism, or are
they independent? The parallel-categories style plot in `1_representation_and_head/04_search`
4.2 is the tool for exactly this question.

**E3 -- confirm the runner-up.** 4.4 confirmed the best trial of each mechanism. Confirm the
second-best trial of `gate` and of `deeper` the same way. Does the gap over its control
survive a configuration the search liked less?

**E4 -- your own dataset, your own specialisation question.** This search space is written
to be reusable as it stands: swap `load_solar_windows` for a loader over a different Monash
series (or your own data), keep everything else, and re-run. Before you look at the result,
decide *why* you expect routing to help or not: does the series contain distinct regimes that
a handful of experts could divide up, or is it one homogeneous process end to end, like
electricity demand? The search tells you whether the bet paid off; only you can say in advance
whether the data gave it anything to win on.

**E5 -- does the chainsaw need a longer budget?** Run `lr_run` for `EPOCHS` of 5, 15 and 45
(the cosine follows `EPOCHS`) with starts 1e-3 and 3e-3. Predict first: at which budget, if
any, does the bigger start end below the smaller one?

**E6 -- grace period and budget.** Run 4.5 with `GRACE` of 1, 3 and 5, and with twice the
trials at the same total epochs. Does the winner of the full search survive each setting, and
does the best score at a fixed budget improve?